# 🔧 OrionFlow — Live Build123d Test Bench

Interactive notebook for testing Build123d-FTC code:
- **Browse** dataset records (text → code pairs)
- **Write** custom Build123d code
- **Execute** and visualize 3D models live
- **Validate** geometry (watertight, volume, bounding box)

### Prerequisites
1. Install VS Code extension: **OCP CAD Viewer** (by bernhard-42)
2. Open viewer: `Ctrl+Shift+P` → `OCP CAD Viewer: Open Viewer`
3. Then run cells below

In [1]:
# ============================================================
# Cell 1: Setup & Imports
# ============================================================
import json
import traceback
from pathlib import Path
from textwrap import dedent

from build123d import *
from ocp_vscode import show, set_defaults

# Configure viewer defaults
set_defaults(
    reset_camera=True,
    axes=True,
    axes0=True,
    grid=(True, True, False),
    transparent=False,
    default_color=(180, 180, 200),
)

print("✅ Build123d + OCP Viewer ready")
print("📌 Make sure OCP CAD Viewer is open: Ctrl+Shift+P → 'OCP CAD Viewer: Open Viewer'")


'reset_camera=True' is deprecated, use 'reset_camera=Camera.RESET' instead

Using port 3939
Jupyter kernel running
Jupyter connection file path written to C:\Users\Legion\.ocpvscode
✅ Build123d + OCP Viewer ready
📌 Make sure OCP CAD Viewer is open: Ctrl+Shift+P → 'OCP CAD Viewer: Open Viewer'


---
## 📂 Mode 1: Browse & Test Dataset Records
Load records from your training dataset and visualize them

In [2]:
# ============================================================
# Cell 2: Load Dataset
# ============================================================
DATASET_PATH = Path("data/build123d_ftc/final/test.jsonl")

records = []
with open(DATASET_PATH) as f:
    for line in f:
        records.append(json.loads(line))

print(f"✅ Loaded {len(records)} records from {DATASET_PATH.name}")
print(f"   Sources: {dict(sorted(((k, v) for k, v in __import__('collections').Counter(r.get('source','?') for r in records).items()), key=lambda x: -x[1]))}")

✅ Loaded 594 records from test.jsonl
   Sources: {'template_edit_param': 142, 'template': 141, 'template_edit_add': 140, 'deepcad_edit_param': 56, 'deepcad': 50, 'deepcad_edit_add': 44, 'complex_generated': 14, 'phase5_ambiguous': 3, 'phase5_impossible': 3, 'phase5_out_of_scope': 1}


In [3]:
# ============================================================
# Cell 3: Pick a Record & Inspect
# ============================================================
# 👉 Change this index to browse different records
RECORD_INDEX = 0

rec = records[RECORD_INDEX]
msgs = rec["messages"]

print(f"═══ Record {RECORD_INDEX}/{len(records)-1} ═══")
print(f"Source:    {rec.get('source', '?')}")
print(f"Edit type: {rec.get('edit_type', '?')}")
print(f"Complexity: {rec.get('complexity', '?')}")
print()

# Show the user prompt
user_msg = next((m["content"] for m in msgs if m["role"] == "user"), "")
print("──── 💬 USER PROMPT ────")
print(user_msg[:1000])
if len(user_msg) > 1000:
    print(f"... [{len(user_msg)} chars total]")
print()

# Extract the assistant's Build123d code
assistant_code = next((m["content"] for m in msgs if m["role"] == "assistant"), "")
print("──── 🔧 BUILD123D CODE ────")
print(assistant_code[:2000])
if len(assistant_code) > 2000:
    print(f"... [{len(assistant_code)} chars total]")

═══ Record 0/593 ═══
Source:    template_edit_add
Edit type: add_feature
Complexity: 3

──── 💬 USER PROMPT ────
Here is my current part:

```python
from build123d import *

# --- Parameters ---
body_d = 10.0  # mm - bushing body OD
bore_d = 5.0  # mm - through bore
body_h = 32.0  # mm
flange_d = 18.0  # mm - flange OD
flange_h = 4.0  # mm - flange thickness

# --- Feature Tree ---
with BuildPart() as part:
    # Feature 1: Flange disc
    with BuildSketch(Plane.XY):
        Circle(flange_d / 2)
    extrude(amount=flange_h)

    # Feature 2: Cylindrical body
    with BuildSketch(Plane.XY.offset(flange_h)):
        Circle(body_d / 2)
    extrude(amount=body_h)

    # Feature 3: Through bore
    with BuildSketch(Plane.XY):
        Circle(bore_d / 2)
    extrude(amount=flange_h + body_h, mode=Mode.SUBTRACT)

# --- Export ---
result = part.part
export_step(result, "output.step")
```

Modification: Add a 0.3mm chamfer to the top edges

──── 🔧 BUILD123D CODE ────
from build123d import *

# --

In [4]:
# ============================================================
# Cell 4: Execute & Visualize the Record
# ============================================================
def execute_and_show(code: str, label: str = "part"):
    """Execute Build123d code and display the resulting 3D model."""
    # Clean the code: remove export_step/export_stl lines
    # (we don't want to write files during testing)
    clean_lines = []
    for line in code.splitlines():
        stripped = line.strip()
        if stripped.startswith("export_step(") or stripped.startswith("export_stl("):
            continue
        clean_lines.append(line)
    clean_code = "\n".join(clean_lines)

    # Execute in a fresh namespace with build123d available
    exec_ns = {}
    exec("from build123d import *", exec_ns)

    try:
        exec(clean_code, exec_ns)
    except Exception as e:
        print(f"❌ EXECUTION ERROR: {type(e).__name__}: {e}")
        traceback.print_exc()
        return None

    # Find the result — look for common variable names
    result = None
    for var_name in ["result", "part", "solid"]:
        if var_name in exec_ns:
            obj = exec_ns[var_name]
            # Handle BuildPart context manager
            if hasattr(obj, "part"):
                result = obj.part
            else:
                result = obj
            break

    if result is None:
        print("⚠️  No 'result', 'part', or 'solid' variable found in code.")
        print(f"   Available vars: {[k for k in exec_ns if not k.startswith('_')]}")
        return None

    # Show in OCP Viewer
    show(result, names=[label])
    print(f"✅ 3D model displayed: '{label}'")

    return result


# Execute the current record
result = execute_and_show(assistant_code, label=f"Record_{RECORD_INDEX}")

+
✅ 3D model displayed: 'Record_0'


In [5]:
# ============================================================
# Cell 5: Validate Geometry
# ============================================================
def validate_geometry(shape, label: str = "part"):
    """Run geometry validation checks on a Build123d shape."""
    print(f"═══ Validation Report: {label} ═══")
    print()

    if shape is None:
        print("❌ No shape to validate")
        return

    # 1. Basic validity
    try:
        is_valid = shape.is_valid()
        print(f"  {'✅' if is_valid else '❌'} Valid geometry: {is_valid}")
    except Exception as e:
        print(f"  ⚠️  Validity check failed: {e}")

    # 2. Volume
    try:
        vol = shape.volume
        print(f"  {'✅' if vol > 0 else '❌'} Volume: {vol:.4f} mm³")
    except Exception as e:
        print(f"  ⚠️  Volume check failed: {e}")

    # 3. Bounding box
    try:
        bb = shape.bounding_box()
        dims = (bb.max.X - bb.min.X, bb.max.Y - bb.min.Y, bb.max.Z - bb.min.Z)
        print(f"  📏 Bounding box: {dims[0]:.2f} × {dims[1]:.2f} × {dims[2]:.2f} mm")
        print(f"     Center: ({(bb.min.X+bb.max.X)/2:.2f}, {(bb.min.Y+bb.max.Y)/2:.2f}, {(bb.min.Z+bb.max.Z)/2:.2f})")

        # Check for degenerate dimensions
        for i, (d, axis) in enumerate(zip(dims, ['X','Y','Z'])):
            if d < 0.01:
                print(f"  ⚠️  {axis}-dimension is near-zero ({d:.4f} mm) — may be flat/degenerate")
    except Exception as e:
        print(f"  ⚠️  Bounding box check failed: {e}")

    # 4. Face/Edge/Vertex count
    try:
        n_faces = len(shape.faces())
        n_edges = len(shape.edges())
        n_verts = len(shape.vertices())
        print(f"  🔺 Topology: {n_faces} faces, {n_edges} edges, {n_verts} vertices")
    except Exception as e:
        print(f"  ⚠️  Topology check failed: {e}")

    # 5. Watertight / solid check via OpenCascade
    try:
        from OCP.BRepCheck import BRepCheck_Analyzer
        analyzer = BRepCheck_Analyzer(shape.wrapped)
        is_watertight = analyzer.IsValid()
        print(f"  {'✅' if is_watertight else '❌'} Watertight (OCC): {is_watertight}")
    except Exception as e:
        print(f"  ⚠️  Watertight check failed: {e}")

    print()
    return True


if result is not None:
    validate_geometry(result, label=f"Record_{RECORD_INDEX}")

═══ Validation Report: Record_0 ═══

  ⚠️  Validity check failed: 'bool' object is not callable
  ✅ Volume: 2822.1712 mm³
  📏 Bounding box: 18.00 × 18.00 × 36.00 mm
     Center: (0.00, 0.00, 18.00)
  🔺 Topology: 8 faces, 13 edges, 8 vertices
  ✅ Watertight (OCC): True



---
## ✏️ Mode 2: Write Custom Build123d Code
Write your own Build123d code and see it live

In [7]:
# ============================================================
# Cell 6: Custom Code — Edit and Run!
# ============================================================

custom_code = """
from build123d import *

# --- Parameters ---
outer_dia = 145.0  # mm
bore_dia = 60.0  # mm - central bore
thickness = 11.0  # mm
n_bolts = 8.0  # bolt count
pcd = 102.0  # mm - bolt circle diameter
bolt_dia = 17.0  # mm - bolt clearance (MODIFIED)

# --- Feature Tree ---
with BuildPart() as part:
    # Feature 1: Circular flange disc
    with BuildSketch(Plane.XY):
        Circle(outer_dia / 2)
    extrude(amount=thickness)

    # Feature 2: Central bore
    with BuildSketch(Plane.XY.offset(thickness)):
        Circle(bore_dia / 2)
    extrude(amount=-thickness, mode=Mode.SUBTRACT)

    # Feature 3: Bolt circle
    with BuildSketch(Plane.XY.offset(thickness)):
        with PolarLocations(pcd / 2, int(n_bolts)):
            Circle(bolt_dia / 2)
    extrude(amount=-thickness, mode=Mode.SUBTRACT)

# --- Result ---
result = part.part
"""

# Execute and display
custom_result = execute_and_show(custom_code, label="Flange_Edit_Test")

if custom_result:
    validate_geometry(custom_result, label="Flange_Edit_Test")

+
✅ 3D model displayed: 'Flange_Edit_Test'
═══ Validation Report: Flange_Edit_Test ═══

  ⚠️  Validity check failed: 'bool' object is not callable
  ✅ Volume: 130566.9469 mm³
  📏 Bounding box: 145.00 × 145.00 × 11.00 mm
     Center: (0.00, 0.00, 5.50)
  🔺 Topology: 12 faces, 30 edges, 20 vertices
  ✅ Watertight (OCC): True



---
## 🔍 Mode 3: Batch Validate Dataset
Scan through multiple records and check which ones produce valid geometry

In [6]:
# ============================================================
# Cell 7: Batch Validation (run on N records)
# ============================================================
N = 20  # Number of records to validate

results_summary = []

for i in range(min(N, len(records))):
    rec = records[i]
    code = next((m["content"] for m in rec["messages"] if m["role"] == "assistant"), "")

    # Strip export lines
    clean = "\n".join(
        l for l in code.splitlines()
        if not l.strip().startswith(("export_step(", "export_stl("))
    )

    exec_ns = {}
    exec("from build123d import *", exec_ns)

    status = {"index": i, "source": rec.get("source", "?")}

    try:
        exec(clean, exec_ns)

        shape = None
        for vn in ["result", "part", "solid"]:
            if vn in exec_ns:
                obj = exec_ns[vn]
                shape = obj.part if hasattr(obj, "part") else obj
                break

        if shape is None:
            status["result"] = "⚠️ no shape"
        elif shape.volume > 0 and shape.is_valid():
            status["result"] = "✅ valid"
            status["volume"] = f"{shape.volume:.1f}"
        else:
            status["result"] = "❌ invalid"
            status["volume"] = f"{shape.volume:.1f}"
    except Exception as e:
        status["result"] = f"💥 {type(e).__name__}"

    results_summary.append(status)
    marker = status["result"].split()[0]
    print(f"  [{i:3d}] {marker} {status['source']:<25} {status.get('volume','')}")

# Summary
valid_count = sum(1 for r in results_summary if "✅" in r["result"])
fail_count = sum(1 for r in results_summary if "❌" in r["result"] or "💥" in r["result"])
warn_count = sum(1 for r in results_summary if "⚠️" in r["result"])
print(f"\n═══ Summary: {valid_count}✅  {fail_count}❌  {warn_count}⚠️  / {len(results_summary)} total ═══")

  [  0] 💥 template_edit_add         
  [  1] 💥 deepcad_edit_param        
  [  2] 💥 template                  
  [  3] 💥 template_edit_param       
  [  4] 💥 template_edit_param       
  [  5] 💥 template_edit_param       
  [  6] 💥 template_edit_add         
  [  7] 💥 template_edit_param       
  [  8] 💥 template_edit_add         
  [  9] 💥 deepcad_edit_add          
  [ 10] 💥 template                  
  [ 11] 💥 template_edit_param       
  [ 12] 💥 deepcad                   
  [ 13] 💥 deepcad_edit_add          
  [ 14] 💥 deepcad_edit_param        
  [ 15] 💥 deepcad_edit_add          
  [ 16] 💥 template                  
  [ 17] 💥 deepcad_edit_add          
  [ 18] 💥 template_edit_add         
  [ 19] 💥 deepcad                   

═══ Summary: 0✅  20❌  0⚠️  / 20 total ═══


---
## 📤 Mode 4: Export a Part
Export the current model to STEP for external viewing

In [ ]:
# ============================================================
# Cell 8: Export to STEP file
# ============================================================
EXPORT_SHAPE = result  # or custom_result
EXPORT_NAME = "live_test_output.step"

if EXPORT_SHAPE is not None:
    export_step(EXPORT_SHAPE, EXPORT_NAME)
    print(f"✅ Exported to {EXPORT_NAME}")
else:
    print("❌ No shape to export — run a code cell first")